### 1. Objetivo del Claim Analyzer

El objetivo de este notebook es desarrollar y validar el componente encargado de transformar una noticia completa en un conjunto de afirmaciones factuales verificables (*claims*).

En el benchmark realizado con AVeriTeC, las claims a verificar ya se proporcionan directamente. Sin embargo, en el sistema final la entrada estará formada por el título y el contenido completo de una noticia, por lo que es necesario identificar previamente las principales afirmaciones que pueden ser contrastadas mediante evidencia externa.

Antes de realizar la extracción de claims, se identifica el idioma de la noticia para determinar si el contenido está escrito en inglés o en español.

El Claim Analyzer utiliza posteriormente un LLM para extraer las principales afirmaciones verificables y devolverlas mediante una salida estructurada que incluye la propia claim, las entidades relevantes y las referencias temporales presentes en el texto.

En esta etapa no se determina si las afirmaciones son verdaderas o falsas ni se realiza búsqueda de evidencias. Las claims extraídas constituirán la entrada del componente de investigación encargado posteriormente de localizar información relevante para su verificación.


### 2. Detección de idioma.

Antes de extraer las claims de una noticia, el sistema identifica el idioma del contenido.

Esta decisión es necesaria porque el sistema final debe procesar noticias tanto en inglés como en español. Además, el idioma condicionará posteriormente algunas partes del flujo, como la aplicación del clasificador de aprendizaje automático en inglés.

Para esta tarea se utiliza la librería `langdetect`, que permite identificar el idioma principal a partir del título y el contenido de la noticia. Con el objetivo de obtener resultados reproducibles, se fija una semilla mediante `DetectorFactory.seed`. Todo este proceso se ha definido en src/claim_analyser.py y aqui probaremos el funcionamiento.

1. Objetivo del Claim Analyzer
2. Detección de idioma
3. Pruebas de language_detector.py
4. Definición de la salida estructurada
5. Prueba de extract_claims()
6. Análisis de claims extraídas
7. Casos problemáticos / ajustes

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [8]:
from src.language_detector import detect_language

### 3. Pruebas de `language_detector.py`

Se realizan pruebas básicas del detector de idioma utilizando una noticia en español y otra en inglés.

El objetivo es comprobar que la función `detect_language()` identifica correctamente los dos idiomas soportados por el sistema antes de conectar este componente con el Claim Analyzer.

In [5]:
title_es = "La Unión Europea prohibirá completamente los pagos en efectivo en 2027"

body_es = """
La medida entrará en vigor en 2027 y afectará a todos los países miembros.
"""

language_es = detect_language(
    title=title_es,
    body=body_es,
)

print("Idioma detectado:", language_es)

Idioma detectado: es


In [6]:
title_en = "The European Union will ban cash payments in 2027"

body_en = """
The measure will come into force in 2027 and will affect all member states.
"""

language_en = detect_language(
    title=title_en,
    body=body_en,
)

print("Idioma detectado:", language_en)

Idioma detectado: en


### 4. Definición de la salida estructurada

El Claim Analyzer debe devolver una salida estructurada que pueda ser utilizada de forma consistente por las siguientes etapas del sistema.

Para ello se definen dos modelos mediante Pydantic:

- `ClaimItem`: representa una afirmación verificable individual extraída de la noticia.
- `ClaimAnalysisResult`: agrupa el conjunto de claims extraídas de una misma noticia.

Cada `ClaimItem` contiene los siguientes campos:

- `id`: identificador numérico de la claim dentro de la noticia.
- `claim`: texto de la afirmación factual y verificable.
- `entities`: entidades relevantes mencionadas en la claim, como personas, organizaciones, países o instituciones.
- `date_reference`: referencia temporal explícita asociada a la claim, cuando exista.

El uso de una salida estructurada permite validar automáticamente que la respuesta generada por el LLM contiene los campos esperados y facilita su utilización posterior por el componente de investigación.

In [10]:
from src.claim_analyzer import (
    ClaimItem,
    ClaimAnalysisResult,
    extract_claims,
)

### 5. Prueba de `extract_claims()`

Una vez validada la detección de idioma y definida la estructura de salida, se prueba el funcionamiento del Claim Analyzer sobre una noticia de ejemplo.

El flujo consiste en detectar primero el idioma del contenido y utilizar posteriormente el título, el cuerpo de la noticia y el idioma detectado como entrada de la función `extract_claims()`.

El objetivo de esta prueba es comprobar que el LLM identifica afirmaciones factuales verificables y devuelve una salida compatible con los modelos Pydantic definidos previamente.

In [11]:
title = "La Unión Europea prohibirá completamente los pagos en efectivo en 2027"

body = """
La Unión Europea ha aprobado una nueva normativa que prohibirá
completamente los pagos en efectivo a partir de 2027.

La medida afectará a todos los Estados miembros.
"""

In [12]:
language = detect_language(
    title=title,
    body=body,
)

print("Idioma detectado:", language)

Idioma detectado: es


In [41]:
from openai import OpenAI
from dotenv import load_dotenv


load_dotenv("../.env")

client = OpenAI()

In [24]:
claim_result = extract_claims(
    title=title,
    body=body,
    language=language,
    client=client,
)

claim_result

ClaimAnalysisResult(claims=[ClaimItem(id=1, claim='La Unión Europea ha aprobado una normativa que prohibirá completamente los pagos en efectivo a partir de 2027.', entities=['Unión Europea'], date_reference='a partir de 2027'), ClaimItem(id=2, claim='La prohibición completa de los pagos en efectivo a partir de 2027 afectará a todos los Estados miembros de la Unión Europea.', entities=['Unión Europea', 'Estados miembros de la Unión Europea'], date_reference='a partir de 2027')])

### 6. Análisis de las claims extraídas

La primera ejecución del Claim Analyzer permite comprobar que el modelo genera una salida estructurada y que las afirmaciones extraídas son, en términos generales, factuales y verificables.

A partir del ejemplo analizado, se revisan varios aspectos relevantes para evaluar la calidad de las claims generadas:

- **Verificabilidad:** las claims deben expresar hechos que puedan contrastarse mediante evidencia externa.
- **Autosuficiencia:** cada claim debe poder entenderse de forma independiente.
- **Redundancia:** se debe evitar repetir información ya incluida en otras claims cuando no sea necesario.
- **Atomicidad:** una claim no debería agrupar demasiados hechos distintos si estos pueden verificarse por separado.
- **Entidades:** las entidades relevantes deben quedar correctamente identificadas.
- **Referencias temporales:** las fechas o periodos mencionados deben conservarse cuando sean relevantes para la verificación.

En la primera prueba, ambas claims son verificables y contienen correctamente las entidades y la referencia temporal. Sin embargo, se observa cierta redundancia entre ellas, ya que la segunda repite parte de la información incluida en la primera.

Esto sugiere que el Claim Analyzer podría beneficiarse de una instrucción más explícita para generar claims menos redundantes y, cuando sea posible, más atómicas.

##### 6.1. Análisis del primer ejemplo

La primera claim extraída es:

> La Unión Europea ha aprobado una normativa que prohibirá completamente los pagos en efectivo a partir de 2027.

Esta claim es verificable y contiene tres `componentes factuales` principales:

- que la Unión Europea ha aprobado una normativa;
- que dicha normativa prohibirá completamente los pagos en efectivo;
- que la medida entrará en vigor a partir de 2027.

La segunda claim es:

> La prohibición completa de los pagos en efectivo a partir de 2027 afectará a todos los Estados miembros de la Unión Europea.

Esta segunda claim también es verificable, pero reutiliza parte de la información ya incluida en la primera claim. El elemento nuevo que aporta es principalmente que la medida afectará a todos los Estados miembros.

Por tanto, el comportamiento observado es correcto desde el punto de vista estructural, aunque existe margen de **mejora** en cuanto a atomicidad y reducción de redundancia.

### 7. Casos problemáticos y ajustes

Después de validar el `funcionamiento básico` del Claim Analyzer, se realizan pruebas adicionales con ejemplos más complejos.

El objetivo es identificar comportamientos que puedan afectar negativamente a las siguientes etapas del sistema, especialmente a la búsqueda de evidencias y a la verificación factual.

Se prestará especial atención a los siguientes aspectos:

- separación de claims compuestas;
- reducción de redundancia entre claims;
- exclusión de opiniones o afirmaciones no verificables;
- conservación de entidades relevantes;
- conservación de referencias temporales;
- capacidad de generar claims comprensibles de forma independiente.

Los ajustes sobre el prompt se realizarán únicamente si los problemas observados aparecen de forma recurrente en varios ejemplos.

In [ ]:
title = "El Gobierno anunció una nueva ley y reducirá los impuestos en 2027"

body = """
El Gobierno anunció este martes una nueva ley económica. La norma reducirá el impuesto sobre la renta en un 10 % a
partir de 2027 y aumentará las ayudas destinadas a pequeñas empresas.
"""

#### 7.1 claim compuesta

In [31]:
language = detect_language(
    title=title,
    body=body,
)

claim_result = extract_claims(
    title=title,
    body=body,
    language=language,
    client=client,
)

claim_result

ClaimAnalysisResult(claims=[ClaimItem(id=1, claim='El Gobierno anunció este martes una nueva ley económica.', entities=['Gobierno'], date_reference='este martes'), ClaimItem(id=2, claim='La nueva ley económica reducirá el impuesto sobre la renta en un 10 % a partir de 2027.', entities=['Gobierno'], date_reference='a partir de 2027'), ClaimItem(id=3, claim='La nueva ley económica aumentará las ayudas destinadas a pequeñas empresas.', entities=['Gobierno', 'pequeñas empresas'], date_reference=None)])

### 7.1. Claim compuesta

En este ejemplo, el Claim Analyzer separa correctamente la información de la noticia en tres afirmaciones distintas:

1. El anuncio de una nueva ley económica.
2. La reducción del impuesto sobre la renta en un 10 % a partir de 2027.
3. El aumento de las ayudas destinadas a pequeñas empresas.

La separación obtenida es adecuada, ya que cada claim representa un hecho verificable diferente. Además, las referencias temporales relevantes se conservan correctamente.

A diferencia del primer ejemplo analizado, en este caso no se observa una redundancia significativa entre las claims. Por tanto, no se considera necesario modificar todavía el prompt únicamente por el comportamiento observado en el ejemplo anterior.

#### 7.2 El siguiente caso que probaría sería uno con hechos mezclados con opinión, para ver si el modelo excluye correctamente frases subjetivas.

In [33]:
title = "El nuevo plan económico del Gobierno es un desastre, según la oposición"

body = """
El Gobierno presentó este jueves un nuevo plan económico que incluye una reducción del impuesto de sociedades del
5 % a partir de 2028.

La oposición calificó el plan como un desastre y afirmó que perjudicará gravemente a las familias.

El Ministerio de Economía indicó que la medida tendrá un coste estimado de 2.000 millones de euros anuales.
"""

In [34]:
language = detect_language(
    title=title,
    body=body,
)

claim_result = extract_claims(
    title=title,
    body=body,
    language=language,
    client=client,
)

claim_result

ClaimAnalysisResult(claims=[ClaimItem(id=1, claim='El Gobierno presentó un nuevo plan económico que incluye una reducción del impuesto de sociedades del 5 % a partir de 2028.', entities=['Gobierno'], date_reference='este jueves; a partir de 2028'), ClaimItem(id=2, claim='El Ministerio de Economía indicó que la reducción del impuesto de sociedades tendrá un coste estimado de 2.000 millones de euros anuales.', entities=['Ministerio de Economía'], date_reference=None)])

### 7.2. Opiniones mezcladas con hechos

En este ejemplo se combina información factual con valoraciones y predicciones subjetivas.

El Claim Analyzer extrae correctamente las afirmaciones verificables relacionadas con:

1. La presentación de un nuevo plan económico que incluye una reducción del impuesto de sociedades del 5 % a partir de 2028.
2. La estimación realizada por el Ministerio de Economía de un coste anual de 2.000 millones de euros.

En cambio, no genera claims a partir de expresiones como *"el plan es un desastre"* o *"perjudicará gravemente a las familias"*, ya que se trata de valoraciones o predicciones atribuidas a la oposición.

Este comportamiento indica que, en este caso, las instrucciones del prompt permiten distinguir adecuadamente entre afirmaciones factuales verificables y contenido subjetivo.

#### 7.3 El siguiente caso que probaría sería el 7.3: entidad implícita o pronombres, porque eso sí puede afectar bastante al Research Agent.

In [35]:
title = "Tesla anuncia una nueva fábrica en México"

body = """
Tesla anunció que construirá una nueva fábrica en México.

# La compañía afirmó que invertirá 5.000 millones de dólares en el proyecto y que la planta comenzará a operar en 2028.

Además, indicó que generará aproximadamente 10.000 empleos directos.
"""

In [36]:
language = detect_language(
    title=title,
    body=body,
)

claim_result = extract_claims(
    title=title,
    body=body,
    language=language,
    client=client,
)

claim_result

ClaimAnalysisResult(claims=[ClaimItem(id=1, claim='Tesla anunció que construirá una nueva fábrica en México.', entities=['Tesla', 'México'], date_reference=None), ClaimItem(id=2, claim='Tesla afirmó que invertirá 5.000 millones de dólares en la nueva fábrica en México.', entities=['Tesla', 'México'], date_reference=None), ClaimItem(id=3, claim='Tesla afirmó que la nueva planta en México comenzará a operar en 2028.', entities=['Tesla', 'México'], date_reference='2028'), ClaimItem(id=4, claim='Tesla indicó que la nueva fábrica en México generará aproximadamente 10.000 empleos directos.', entities=['Tesla', 'México'], date_reference=None)])

### 7.3. Entidades implícitas y referencias anafóricas

En este ejemplo se evalúa si el Claim Analyzer es capaz de resolver referencias implícitas como *"la compañía"* o *"la planta"* y generar claims comprensibles de forma independiente.

El resultado obtenido muestra que el modelo sustituye correctamente estas referencias por entidades explícitas, principalmente `Tesla` y `México`.

Además, la información se separa en cuatro afirmaciones distintas:

1. La construcción de una nueva fábrica de Tesla en México.
2. La inversión de 5.000 millones de dólares en el proyecto.
3. El inicio de operaciones de la planta en 2028.
4. La generación aproximada de 10.000 empleos directos.

Las claims generadas son autosuficientes y mantienen las entidades relevantes y la referencia temporal cuando corresponde.

Este comportamiento es adecuado para las etapas posteriores del sistema, ya que reduce la ambigüedad en las búsquedas de evidencia.

#### 7.4 El siguiente paso es probar un caso realmente conflictivo o con varios componentes factuales en una misma frase, para ver si el modelo decide bien cuándo separar y cuándo mantener contexto.

In [37]:
title = "El Gobierno ampliará las ayudas y reducirá el IVA de la electricidad en 2027"

body = """
El Gobierno anunció que en 2027 ampliará las ayudas destinadas a hogares vulnerables y reducirá el IVA de la 
electricidad del 21 % al 10 %.

La medida estará incluida en los próximos Presupuestos Generales del Estado.
"""

In [38]:
language = detect_language(
    title=title,
    body=body,
)

claim_result = extract_claims(
    title=title,
    body=body,
    language=language,
    client=client,
)

claim_result

ClaimAnalysisResult(claims=[ClaimItem(id=1, claim='El Gobierno ampliará en 2027 las ayudas destinadas a hogares vulnerables.', entities=['Gobierno'], date_reference='2027'), ClaimItem(id=2, claim='El Gobierno reducirá en 2027 el IVA de la electricidad del 21 % al 10 %.', entities=['Gobierno'], date_reference='2027'), ClaimItem(id=3, claim='La ampliación de ayudas a hogares vulnerables y la reducción del IVA de la electricidad estarán incluidas en los próximos Presupuestos Generales del Estado.', entities=['Presupuestos Generales del Estado'], date_reference=None)])

### 7.4. Claims con varios componentes factuales relacionados

En este ejemplo se evalúa la capacidad del Claim Analyzer para separar distintos hechos contenidos en una misma frase sin perder el contexto necesario.

El modelo extrae correctamente tres afirmaciones:

1. La ampliación en 2027 de las ayudas destinadas a hogares vulnerables.
2. La reducción en 2027 del IVA de la electricidad del 21 % al 10 %.
3. La inclusión de ambas medidas en los próximos Presupuestos Generales del Estado.

Las dos primeras claims se separan adecuadamente porque representan medidas distintas y verificables de forma independiente.

La tercera claim mantiene juntas ambas medidas, lo que resulta razonable porque el hecho a verificar es precisamente que las dos estarán incluidas en los próximos Presupuestos Generales del Estado.

El resultado muestra un equilibrio adecuado entre atomicidad y conservación del contexto.

#### 7.5 haremos un último caso 7.5 más difícil, por ejemplo con fechas relativas, atribuciones y una claim que dependa de contexto, antes de decidir si dejamos el prompt como está o hacemos algún ajuste.

In [39]:
title = "La ministra anunció que las nuevas ayudas comenzarán el próximo mes"

body = """
La ministra de Vivienda, Marta López, anunció este miércoles que el Gobierno pondrá en marcha un nuevo programa de 
ayudas al alquiler.

Según explicó, las ayudas comenzarán el próximo mes y estarán dirigidas a jóvenes menores de 30 años.

El programa tendrá un presupuesto inicial de 500 millones de euros.
"""

In [40]:
language = detect_language(
    title=title,
    body=body,
)

claim_result = extract_claims(
    title=title,
    body=body,
    language=language,
    client=client,
)

claim_result

ClaimAnalysisResult(claims=[ClaimItem(id=1, claim='La ministra de Vivienda, Marta López, anunció un nuevo programa de ayudas al alquiler que el Gobierno pondrá en marcha.', entities=['Marta López', 'Ministerio de Vivienda', 'Gobierno'], date_reference='este miércoles'), ClaimItem(id=2, claim='El nuevo programa de ayudas al alquiler comenzará el próximo mes.', entities=['Gobierno'], date_reference='el próximo mes'), ClaimItem(id=3, claim='Las ayudas al alquiler estarán dirigidas a jóvenes menores de 30 años.', entities=['Gobierno'], date_reference=None), ClaimItem(id=4, claim='El nuevo programa de ayudas al alquiler tendrá un presupuesto inicial de 500 millones de euros.', entities=['Gobierno'], date_reference=None)])

### 7.5. Atribución, contexto temporal y fechas relativas

En este ejemplo se evalúa la capacidad del Claim Analyzer para gestionar atribuciones, referencias anafóricas y expresiones temporales relativas.

El modelo genera cuatro claims diferenciadas:

1. El anuncio de un nuevo programa de ayudas al alquiler por parte de la ministra de Vivienda, Marta López.
2. El inicio del programa el próximo mes.
3. La orientación de las ayudas a jóvenes menores de 30 años.
4. El presupuesto inicial de 500 millones de euros.

El resultado muestra que el modelo resuelve correctamente la referencia a la ministra y mantiene entidades explícitas en las claims generadas. Además, las afirmaciones son autosuficientes y representan hechos verificables de forma independiente.

Las referencias temporales relativas, como *"este miércoles"* y *"el próximo mes"*, se conservan correctamente en el campo `date_reference`. No obstante, estas expresiones no se normalizan a fechas absolutas, por lo que esta cuestión podría considerarse en una versión posterior del sistema si fuera necesario para mejorar las búsquedas temporales.

### 7.6. Conclusión de las pruebas

Las pruebas realizadas muestran que la versión actual del `Claim Analyzer` presenta un comportamiento consistente en distintos tipos de noticias.

En los ejemplos analizados, el sistema:

- separa adecuadamente afirmaciones con varios componentes factuales;
- evita convertir opiniones o valoraciones subjetivas en claims verificables;
- resuelve correctamente referencias implícitas y pronombres;
- conserva las entidades relevantes;
- mantiene las referencias temporales asociadas a cada afirmación;
- genera claims suficientemente autosuficientes para ser utilizadas por las siguientes etapas del sistema.

Aunque en el primer ejemplo se observó cierta redundancia entre claims, este comportamiento no se repitió de forma sistemática en los casos posteriores.

Por este motivo, no se realizan ajustes adicionales sobre el prompt en esta fase y se mantiene la versión actual como configuración inicial del Claim Analyzer.